# 🚀 CRANE-VAD: Counterfactual Residual Alignment Network with Evidential Fusion
### 100% Standalone Weakly-Supervised Video Anomaly Detection (WS-VAD)
**Tác giả:** Huỳnh Quang Vũ (MSSV: 2351010247)  
**Giảng viên hướng dẫn:** TS. Lê Viết Tuấn  
**Đơn vị:** Trường Đại học Mở TP. Hồ Chí Minh (HCMCOU)  

---
### 🎯 Mục tiêu khoa học & Đột phá kiến trúc:
1. **100% Train from Scratch (Bỏ hoàn toàn Warm-Start):** Không phụ thuộc vào bất kỳ checkpoint pre-trained nào của DSANet.
2. **Label-Gated Asymmetric Memory (LGAM):** Triệt tiêu hoàn toàn hiện tượng nhiễm độc nguyên mẫu (Prototype Contamination) bằng cách chỉ cho phép video Y=0 cập nhật Prototype P.
3. **Bidirectional S4D Backbone (O(T)):** Nắm bắt ngữ cảnh thời gian liên tục với độ phức tạp tuyến tính, giải phóng GPU VRAM.
4. **Counterfactual Residual-to-Text Alignment:** Triệt tiêu >90% nhiễu phông nền tĩnh trước khi đối sánh với text prompt CLIP.
5. **Beta-Evidential Gate:** Cổng hợp nhất dựa trên độ bất định Bayesian (Inverse-Variance), loại bỏ hoàn toàn các mẹo khởi tạo bias âm heuristic.

In [ ]:
# =====================================================================
# 1. CLONE MÃ NGUỒN & THIẾT LẬP MÔI TRƯỜNG DUAL TESLA T4
# =====================================================================
import os, sys, time, subprocess, shutil
from pathlib import Path
import torch

REPO_URL = 'https://github.com/Sharvuz/CRANE-VAD.git'
if not os.path.exists('CRANE-VAD') and not os.path.exists('src/model.py'):
    print(f'📥 Đang tải mã nguồn từ {REPO_URL}...')
    subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('CRANE-VAD')
    print('Đã chuyển vào thư mục CRANE-VAD!')
elif os.path.exists('CRANE-VAD'):
    os.chdir('CRANE-VAD')

print('=== KIỂM TRA CẤU HÌNH PHẦN CỨNG ===')
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
num_gpus = torch.cuda.device_count()
print(f'Số lượng GPU khả dụng: {num_gpus}')
for i in range(num_gpus):
    print(f'  [GPU {i}] {torch.cuda.get_device_name(i)} | VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB')

if num_gpus > 1:
    print('🔥 ĐÃ KÍCH HOẠT CHẾ ĐỘ DUAL GPU (DataParallel)!')

# Cài đặt các thư viện cần thiết
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'scikit-learn', 'pandas', 'tqdm'])
print('Môi trường đã sẵn sàng!')


In [ ]:
# =====================================================================
# 2. TỰ ĐỘNG QUÉT & LIÊN KẾT DỮ LIỆU TỪ KAGGLE INPUTS
# =====================================================================
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')

print('🔍 Đang quét toàn bộ file đặc trưng .npy trên Kaggle...')
all_npy = list(INPUT.rglob('*.npy')) + list(WORK.rglob('*.npy'))
print(f'Đã tìm thấy tổng cộng: {len(all_npy)} file .npy')

# Tiền xử lý text embeddings
!python scripts/prepare_text_embeddings.py
print('Đã chuẩn bị xong CLIP Text Embeddings cho UCF và XD!')

---
## 🏛️ PHẦN 1: HUẤN LUYỆN CRANE-VAD TRÊN UCF-CRIME
- **Mục tiêu:** ROC-AUC >= 88.5% | AP >= 38.5%
- **Cơ chế:** LGAM đảm bảo không nhiễm độc từ Epoch 0.

In [ ]:
# =====================================================================
# 3. HUẤN LUYỆN TRÊN UCF-CRIME (10 EPOCHS - EVAL MỖI EPOCH)
# =====================================================================
!python -u -m src.train_ucf \
    --dataset ucf \
    --batch-size 64 \
    --epochs 10 \
    --lr 1e-4 \
    --lr-proto 5e-4 \
    --n-prototypes 16 \
    --eval-freq 1 \
    --model-save-dir models/ucf


In [ ]:
# =====================================================================
# 4. ĐÁNH GIÁ CHI TIẾT TẬP TEST UCF-CRIME
# =====================================================================
!python -u -m src.test_ucf \
    --dataset ucf \
    --checkpoint models/ucf/crane_ucf_best.pth

---
## 🏛️ PHẦN 2: HUẤN LUYỆN CRANE-VAD TRÊN XD-VIOLENCE
- **Mục tiêu:** AP >= 86.5%
- **Cơ chế:** Tận dụng Dual Tesla T4 với Batch Size 96 hoặc 128.

In [ ]:
# =====================================================================
# 5. HUẤN LUYỆN TRÊN XD-VIOLENCE (15 EPOCHS - EVAL MỖI EPOCH)
# =====================================================================
!python -u -m src.train_xd \
    --dataset xd \
    --batch-size 96 \
    --epochs 15 \
    --lr 1e-4 \
    --lr-proto 5e-4 \
    --n-prototypes 16 \
    --eval-freq 1 \
    --model-save-dir models/xd


In [ ]:
# =====================================================================
# 6. ĐÁNH GIÁ CHI TIẾT TẬP TEST XD-VIOLENCE
# =====================================================================
!python -u -m src.test_xd \
    --dataset xd \
    --checkpoint models/xd/crane_xd_best.pth

In [ ]:
# =====================================================================
# 7. HOÀN THÀNH & TÓM TẮT THỰC NGHIỆM
# =====================================================================
print('🎉 Đã hoàn thành toàn bộ chu trình huấn luyện & đánh giá CRANE-VAD trên Dual Tesla T4!')
